In [ ]:
import torch
import torch.nn as nn

model = nn.Sequential(nn.Linear(10, 100), nn.ReLU(), nn.Linear(100, 1))
# [...] pretend the 32-bit model is trained here
model.half()  # convert the model parameters to half precision (16 bits)

In [ ]:
X = torch.rand(3, 10, dtype=torch.float16)  # some 16-bit input
y_pred = model(X)  # 16-bit output

In [ ]:
model = nn.Sequential(nn.Linear(10, 100, dtype=torch.float16), nn.ReLU(),
                      nn.Linear(100, 1, dtype=torch.float16))

In [ ]:
from torch.amp import GradScaler

device = "cuda" if torch.cuda.is_available() else "cpu"

def train_mpt(model, optimizer, criterion, train_loader, n_epochs,
              dtype=torch.float16, init_scale=2.0**16):
    grad_scaler = GradScaler(device=device, init_scale=init_scale)
    model.train()
    for epoch in range(n_epochs):
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            with torch.autocast(device_type=device, dtype=dtype):
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
            grad_scaler.scale(loss).backward()
            grad_scaler.step(optimizer)
            grad_scaler.update()
            optimizer.zero_grad()

In [1]:
import platform

machine = platform.machine().lower()
engine = "qnnpack" if ("arm" in machine or "aarch64" in machine) else "x86"

In [ ]:
from torch.ao.quantization import quantize_dynamic

model = nn.Sequential(nn.Linear(10, 100), nn.ReLU(), nn.Linear(100, 1))
# [...] pretend the 32-bit model is trained here
torch.backends.quantized.engine = engine
qmodel = quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
X = torch.randn(3, 10)
y_pred = qmodel(X)  # float inputs and outputs, but quantized internally

In [ ]:
# from torch.ao.quantization import get_default_qconfig, QuantStub, DeQuantStub
#
# model = nn.Sequential(QuantStub(),
#                       nn.Linear(10, 100), nn.ReLU(), nn.Linear(100, 1),
#                       DeQuantStub())
# # [...] pretend the 32-bit model is trained here
# model.qconfig = get_default_qconfig(engine)
# torch.ao.quantization.prepare(model, inplace=True)
# for X_batch, _ in calibration_loader:
#     model(X_batch)
# torch.ao.quantization.convert(model, inplace=True)

In [ ]:
# from torch.ao.quantization import get_default_qat_qconfig
#
# model = nn.Sequential(nn.Linear(10, 100), nn.ReLU(), nn.Linear(100, 1))
# model.qconfig = get_default_qat_qconfig(engine)
# torch.ao.quantization.prepare_qat(model, inplace=True)
# train(model, optimizer, [...])  # train the model normally
# torch.ao.quantization.convert(model.eval(), inplace=True)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", quantization_config=bnb_config)